In [3]:
# %%
# =============================================================================
# Paper C · 04_llm_code_enrich  (FINAL)
# RQ3 (redefined): LLM-based SEMANTIC ENRICHMENT of coded lawsuit fields.
#
# SMEsD pickles hold only integer codes (cause 0-10, court 0-3, category 0-3) —
# NO raw judgment text. But SMEsD.md ships a full CODEBOOK. We feed each code's
# real meaning (Chinese原文, no translation) to an LLM and build a
# code -> [1536-d semantic embedding | 1-d risk-direction] lookup:
#
#   • semantic embedding : LLM description of the code's bankruptcy-risk
#                          implication, embedded (captures inter-code similarity)
#   • risk-direction (A) : signed scalar in [-1,+1] the description embedding
#                          under-separates (verified: category plaintiff-won ↔
#                          defendant-failed cosine 0.748). Extracted explicitly
#                          so 05 can use DIRECTION that counts/embeddings miss.
#
# Builds + validates only (no performance numbers — those come in 05).
# All LLM calls (19 codes × 2) are CACHED + temperature 0 → re-runs are free
# and deterministic.
#
# Inputs : .env (OPENAI_API_KEY, LLM_MODEL_MID)   +   SMEsD.md codebook (inlined)
# Outputs: data/processed/code_emb_lookup.pkl      (code -> 1537-d)   ← 05 uses this
#          data/processed/code_semantic_emb.pkl    (full generation cache)
#          data/processed/code_direction.pkl       (direction cache)
#          artifacts/tables/04_code_similarity.csv (human-readable table)
# =============================================================================
import os, re, hashlib, warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

# --- Project scaffolding (matches 03 conventions) ----------------------------
ROOT = Path.cwd()
if ROOT.name in {"notebooks", "paperA", "paperB", "paperC", "paperD"}:
    ROOT = ROOT.parents[0] if ROOT.name == "notebooks" else ROOT.parents[1]
PROCESSED = ROOT / "data" / "processed"
INFER = ROOT / "artifacts" / "inference"
TAB = ROOT / "artifacts" / "tables"
for p in (PROCESSED, INFER, TAB):
    p.mkdir(parents=True, exist_ok=True)

def rel(p):
    try: return str(Path(p).resolve().relative_to(ROOT.resolve()))
    except ValueError: return Path(p).name

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("[ERROR] OPENAI_API_KEY not set. Add it to .env.")
client = OpenAI(api_key=OPENAI_API_KEY)

GEN_MODEL = os.getenv("LLM_MODEL_MID", "gpt-5.4-mini")   # description + direction
EMBED_MODEL = "text-embedding-3-small"                    # 1536-d embeddings
TEMPERATURE = 0.0
SEED = 42
np.random.seed(SEED)
print(f"[INFO] gen={GEN_MODEL}  embed={EMBED_MODEL}  temp={TEMPERATURE}")


# %%
# --- 1. Codebook (verbatim from SMEsD.md; Chinese原文 kept, NOT translated) ----
CAUSE = {
    0: "金融借款合同纠纷 / 借款合同纠纷 / 小额借款合同纠纷 / 金融不良债权追偿纠纷 / 企业借贷纠纷 / 民间借贷纠纷 / 追偿权纠纷 (金融·借贷类)",
    1: "买卖合同纠纷 / 买卖合同 (买卖类)",
    2: "租赁合同纠纷 (租赁类)",
    3: "工程合同纠纷 / 建设工程施工合同纠纷 / 建设工程分包合同纠纷 / 加工合同纠纷 / 运输合同纠纷 (工程·承揽·运输类)",
    4: "债权纠纷 / 债权转让合同纠纷 / 债务转移合同纠纷 / 债权人代位权纠纷 (债权类)",
    5: "其他合同 / 合同纠纷 / 不当得利纠纷 / 商品房预售合同纠纷 / 劳务合同纠纷 / 建设工程合同纠纷 / 物业服务合同纠纷 / 商品房销售合同纠纷 / 旅游合同纠纷 / 委托合同纠纷 / 劳动合同纠纷 (其他合同类)",
    6: "知识产权与竞争纠纷 / 知识产权合同纠纷 / 知识产权权属、侵权纠纷 / 侵权纠纷 / 不正当竞争纠纷 / 侵害商标权纠纷 (知识产权·竞争类)",
    7: "劳动争议、人事争议 / 劳动争议 / 人事争议 / 追索劳动报酬纠纷 (劳动·人事类)",
    8: "侵权责任纠纷 / 事故责任纠纷 (侵权责任类)",
    9: "与公司、证券、保险、票据等有关的民事纠纷 / 与企业有关的纠纷 / 与公司有关的纠纷 / 与破产有关的纠纷 / 期货交易纠纷 / 信托纠纷 / 保险纠纷 / 票据纠纷 / 信用证纠纷 / 股东损害公司债权人利益责任纠纷 / 职工破产债权确认纠纷 / 股权转让纠纷 / 破产债权确认纠纷 / 破产撤销权纠纷 / 债权人撤销权纠纷 / 普通破产债权确认纠纷 / 提供劳务者受害责任纠纷 / 追收未缴出资纠纷 (公司·证券·破产类)",
    10: "海事海商纠纷 / 通海水域污染损害责任纠纷 / 物权保护纠纷 / 所有权纠纷 / 用益物权纠纷 / 担保物权纠纷 / 物权纠纷 / 损害赔偿纠纷 / 婚姻家庭纠纷 / 继承纠纷 / 人格权纠纷 (海事·物权·其他类)",
}
COURT = {
    0: "基层人民法院 (Grassroots people's court) — 最低审级",
    1: "中级人民法院 (Intermediate people's court)",
    2: "高级人民法院 (Higher people's court)",
    3: "最高人民法院 (Supreme court) — 最高审级",
}
CATEGORY = {
    0: "本企业作为原告胜诉 (plaintiff won)",
    1: "本企业作为原告败诉 (plaintiff failed)",
    2: "本企业作为被告胜诉 (defendant won)",
    3: "本企业作为被告败诉 (defendant failed)",
}
FIELDS = {"cause": CAUSE, "court": COURT, "category": CATEGORY}
FIELD_FRAME = {
    "cause": "一起诉讼的『案由类型』",
    "court": "审理诉讼的『法院层级』",
    "category": "企业在诉讼中的『判决结果』",
}
N_TOTAL = sum(len(d) for d in FIELDS.values())
print(f"[INFO] codebook: cause={len(CAUSE)} court={len(COURT)} "
      f"category={len(CATEGORY)}  total={N_TOTAL} codes")


# %%
# --- 2. Prompt builders ------------------------------------------------------
def desc_prompt(field, meaning):
    return (
        f"你是中国中小企业信用风险分析专家。以下是{FIELD_FRAME[field]}：\n"
        f"『{meaning}』\n\n"
        f"请仅就『该属性对企业破产/违约风险的含义』给出简明分析，"
        f"包括：(1) 该属性通常反映的财务或经营压力；"
        f"(2) 相对而言指向更高还是更低的破产风险，及原因。"
        f"控制在80字以内，客观中立，不要复述题目。"
    )

def dir_prompt(field, meaning, description):
    return (
        f"你是中国中小企业信用风险分析专家。\n"
        f"属性类别：{FIELD_FRAME[field]}\n"
        f"具体含义：『{meaning}』\n"
        f"已知风险分析：『{description}』\n\n"
        f"请给出该属性对企业『破产/违约风险』的方向性评分，"
        f"取值范围 -1.0 到 +1.0：\n"
        f"  +1.0 = 强烈提高破产风险；  0 = 中性/不确定；  -1.0 = 强烈降低破产风险。\n"
        f"只输出一个数字（可带小数和正负号），不要任何其他文字。"
    )

def parse_score(text):
    m = re.search(r"[-+]?\d*\.?\d+", text)
    if not m:
        return 0.0
    return max(-1.0, min(1.0, float(m.group())))


# %%
# --- 3. Cached LLM calls: description → embedding → direction -----------------
SEM_CACHE = PROCESSED / "code_semantic_emb.pkl"
DIR_CACHE = PROCESSED / "code_direction.pkl"

def load_pkl(p):
    return pd.read_pickle(p) if p.exists() else {}

def sem_key(field, code, meaning):
    raw = f"{GEN_MODEL}|{EMBED_MODEL}|{TEMPERATURE}|{field}|{code}|{meaning}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:16]

def gen_description(prompt):
    r = client.chat.completions.create(
        model=GEN_MODEL, temperature=TEMPERATURE,
        messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content.strip()

def embed(text):
    r = client.embeddings.create(model=EMBED_MODEL, input=text)
    return np.array(r.data[0].embedding, dtype=np.float32)

def gen_direction(prompt):
    r = client.chat.completions.create(
        model=GEN_MODEL, temperature=TEMPERATURE,
        messages=[{"role": "user", "content": prompt}])
    return parse_score(r.choices[0].message.content.strip())

sem_cache, dir_cache = load_pkl(SEM_CACHE), load_pkl(DIR_CACHE)
records = {}            # (field, code) -> dict(meaning, description, embedding, key)
directions = {}         # (field, code) -> float
made_s = hit_s = made_d = hit_d = 0

for field, codes in FIELDS.items():
    for code, meaning in codes.items():
        # 3a. description + semantic embedding (cached)
        k = sem_key(field, code, meaning)
        if k in sem_cache:
            rec = sem_cache[k]; hit_s += 1
        else:
            desc = gen_description(desc_prompt(field, meaning))
            rec = {"field": field, "code": code, "meaning": meaning,
                   "description": desc, "embedding": embed(desc), "key": k}
            sem_cache[k] = rec; made_s += 1
        records[(field, code)] = rec

        # 3b. explicit risk-direction scalar (cached, improvement A)
        dk = f"{GEN_MODEL}|{TEMPERATURE}|dir|{field}|{code}|{rec['key']}"
        if dk in dir_cache:
            directions[(field, code)] = dir_cache[dk]; hit_d += 1
        else:
            score = gen_direction(dir_prompt(field, meaning, rec["description"]))
            directions[(field, code)] = score; dir_cache[dk] = score; made_d += 1

pd.to_pickle(sem_cache, SEM_CACHE)
pd.to_pickle(dir_cache, DIR_CACHE)
print(f"[INFO] descriptions: {made_s} new / {hit_s} cached → {rel(SEM_CACHE)}")
print(f"[INFO] directions  : {made_d} new / {hit_d} cached → {rel(DIR_CACHE)}")


# %%
# --- 4. Inspect generated descriptions ---------------------------------------
for field in ["cause", "court", "category"]:
    print(f"\n===== {field} =====")
    for code in sorted(FIELDS[field]):
        print(f"  [{code}] {records[(field, code)]['description']}")


# %%
# --- 5. Inspect direction scores ---------------------------------------------
print("risk-direction scores (+1 raises bankruptcy risk, -1 lowers):")
for field in ["cause", "court", "category"]:
    print(f"\n  {field}:")
    for code in sorted(FIELDS[field]):
        d = directions[(field, code)]
        bar = "█" * int(round((d + 1) * 10))
        print(f"    [{code}] {d:+.2f}  {bar}  {records[(field,code)]['meaning'][:32]}")


# %%
# --- 6. Sanity checks: semantics + direction vs domain logic -----------------
def cos(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

# 6a. semantic-embedding probes (expected: partial separation only — this is
#     WHY the explicit direction scalar exists)
c_solvency = cos(records[("cause", 0)]["embedding"], records[("cause", 9)]["embedding"])
cat_opp = cos(records[("category", 0)]["embedding"], records[("category", 3)]["embedding"])
print(f"[semantic] cause 0↔9 (both solvency) cosine = {c_solvency:.3f}")
print(f"[semantic] category won↔lost cosine       = {cat_opp:.3f} "
      f"(under-separated → direction scalar compensates)")

# 6b. direction probes (expected: CLEAN separation)
cat = {c: directions[("category", c)] for c in range(4)}
court = {c: directions[("court", c)] for c in range(4)}
print(f"\n[direction] category: won(0)={cat[0]:+.2f} pl-lost(1)={cat[1]:+.2f} "
      f"def-won(2)={cat[2]:+.2f} def-lost(3)={cat[3]:+.2f}")
print(f"            → wins {'<0' if max(cat[0],cat[2])<0 else '?'}, "
      f"losses {'>0' if min(cat[1],cat[3])>0 else '?'} "
      f"(gap won↔def-lost = {cat[3]-cat[0]:+.2f})")
print(f"[direction] court (expect ↑ with seniority): "
      f"{[round(court[c],2) for c in range(4)]}")


# %%
# --- 7. Final lookup: [1536-d semantic | 1-d direction] = 1537-d --------------
emb_lookup = {}
for (field, code), rec in records.items():
    sem = rec["embedding"].astype(np.float32)              # 1536-d
    d = np.float32(directions[(field, code)])              # 1-d
    emb_lookup[(field, code)] = np.concatenate([sem, [d]])  # 1537-d
DIM = next(iter(emb_lookup.values())).shape[0]
pd.to_pickle(emb_lookup, PROCESSED / "code_emb_lookup.pkl")
print(f"→ saved {rel(PROCESSED / 'code_emb_lookup.pkl')}  "
      f"({N_TOTAL} codes × {DIM}-d = 1536 semantic + 1 direction)")

# human-readable table
tbl = pd.DataFrame([{
    "field": f, "code": c, "meaning": records[(f, c)]["meaning"],
    "description": records[(f, c)]["description"],
    "risk_direction": directions[(f, c)], "emb_dim": DIM,
} for (f, c) in records]).sort_values(["field", "code"]).reset_index(drop=True)
tbl.to_csv(TAB / "04_code_similarity.csv", index=False)
print(f"→ saved {rel(TAB / '04_code_similarity.csv')}")


# %%
# --- 8. Notes for 05_gnn_plus_enrich -----------------------------------------
print(f"""
04 complete — semantic + direction code features ready (RQ3 redefined)
======================================================================
  code_emb_lookup.pkl : (field, code) -> {DIM}-d  [1536 semantic | 1 direction]

  Next (05_gnn_plus_enrich), per company:
    • pool its lawsuit events into a fixed profile —
        semantic: mean-pool the 1536-d vectors (captures typical case mix)
        direction: multi-stat (max / min / sum / mean) so opposing signals do
                   NOT cancel (a firm with both plaintiff-wins and defendant-
                   losses keeps BOTH the -0.70 and +0.80 extremes)
    • concat to the 23-d company features; re-run the 03 hetero-GNN UNCHANGED
      (same arch/HP, only inputs differ) and report ΔAUC vs 03.

  Value-add hypothesis: DIRECTION (defendant-failed +0.80 vs plaintiff-won
  -0.70) is what raw counts and even the description embedding miss; uplift
  should concentrate in firms whose lawsuit mix is directionally informative.
""")
print("04_llm_code_enrich (FINAL) complete.")

[INFO] gen=gpt-5.4-mini  embed=text-embedding-3-small  temp=0.0
[INFO] codebook: cause=11 court=4 category=4  total=19 codes
[INFO] descriptions: 6 new / 13 cached → data\processed\code_semantic_emb.pkl
[INFO] directions  : 6 new / 13 cached → data\processed\code_direction.pkl

===== cause =====
  [0] 通常反映融资周转紧张、现金流承压或历史债务未清。相较一般经营纠纷，更指向较高破产/违约风险，因其直接关联偿债能力与外部融资可得性。
  [1] 通常反映应收账款回款、供货履约或库存周转压力，说明交易链条可能承压。相对而言偏向更高破产/违约风险，因持续买卖纠纷常意味着现金流紧张、合作稳定性下降。
  [2] 通常反映租金、押金或履约争议，说明现金流、合同履约或资产使用压力较大。相对而言偏向更高破产/违约风险，因持续经营稳定性与支付能力可能受影响。
  [3] 多见于工程回款慢、垫资重、履约链条长，易致现金流紧张。相较一般商事纠纷，通常指向较高破产/违约风险，因项目周期长、应收账款和成本超支压力更大。
  [4] 通常反映应收债权流转、债务重组或回款受阻，说明资金周转和履约压力较大。相对而言，指向较高破产风险，因其往往伴随现金流紧张、债务链条复杂及违约纠纷增多。
  [5] 多属日常经营或履约争议，常反映回款、工程、用工或服务结算压力，可能占用现金流。相较纯商业扩张类纠纷，通常指向中等偏高的违约/破产风险，但单案不足以定论。
  [6] 通常反映品牌、渠道或技术受冲击，可能带来赔偿、禁令及销售下滑压力。相对而言指向较高破产/违约风险，因核心经营稳定性和现金流更易受扰动。
  [7] 通常反映工资、社保或补偿金支付压力，说明现金流或用工管理可能承压。相对而言偏向更高破产/违约风险，因其常与短期资金紧张、经营波动或劳动纠纷累积有关。
  [8] 通常反映安全、管理或赔付压力，可能带来一次性赔偿、停工及声誉损失。相对而言多指向较高破产风险，因突发负债和现金流冲击较直接。
  [9]

In [1]:
# %%
# --- FREEZE 04 embeddings: verify + lock, NO API calls -----------------------
# Loads the existing lookup, checks all 19 codes are present and correctly shaped
# (1537-d = 1536 semantic + 1 direction), records a content hash, and writes a
# frozen copy that 05/06 will load. If anything is missing/malformed it STOPS so
# you can re-run 04 once more — but it never calls the API here.
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name in {"notebooks", "paperA", "paperB", "paperC", "paperD"}:
    ROOT = ROOT.parents[0] if ROOT.name == "notebooks" else ROOT.parents[1]
PROCESSED = ROOT / "data" / "processed"

lookup = pd.read_pickle(PROCESSED / "code_emb_lookup.pkl")

# 1. structural checks (expected: 19 codes, all 1537-d)
EXPECTED = ([("cause", c) for c in range(11)]
            + [("court", c) for c in range(4)]
            + [("category", c) for c in range(4)])
missing = [k for k in EXPECTED if k not in lookup]
assert not missing, f"missing codes: {missing} — re-run 04 once, then freeze"
dims = {k: np.asarray(v).shape for k, v in lookup.items()}
bad = {k: s for k, s in dims.items() if s != (1537,)}
assert not bad, f"wrong shapes: {bad} — re-run 04 once, then freeze"
print(f"structure OK: {len(lookup)} codes, all 1537-d (1536 semantic + 1 direction)")

# 2. content hash: fingerprint the exact float contents so we can detect drift
def content_hash(lk):
    h = hashlib.sha256()
    for k in sorted(lk.keys()):                 # deterministic key order
        h.update(str(k).encode())
        h.update(np.asarray(lk[k], dtype=np.float32).tobytes())
    return h.hexdigest()[:16]

FROZEN_HASH = content_hash(lookup)
print(f"content hash: {FROZEN_HASH}")

# 3. write a frozen copy + its hash (05/06 load THIS, not the live cache)
pd.to_pickle(lookup, PROCESSED / "code_emb_lookup.FROZEN.pkl")
(PROCESSED / "code_emb_lookup.FROZEN.sha256").write_text(FROZEN_HASH)
print("frozen: code_emb_lookup.FROZEN.pkl (+ .sha256)")


# %%
# --- Guard cell: re-run anytime to confirm the frozen file hasn't drifted -----
# Use this at the top of 05/06 to assert you're on the frozen embeddings.
def assert_frozen():
    lk = pd.read_pickle(PROCESSED / "code_emb_lookup.FROZEN.pkl")
    want = (PROCESSED / "code_emb_lookup.FROZEN.sha256").read_text().strip()
    got = content_hash(lk)
    assert got == want, f"FROZEN embeddings changed! {got} != {want}"
    print(f"frozen embeddings verified ({got}); {len(lk)} codes")
    return lk

_ = assert_frozen()

structure OK: 19 codes, all 1537-d (1536 semantic + 1 direction)
content hash: 42b542915055e17c
frozen: code_emb_lookup.FROZEN.pkl (+ .sha256)
frozen embeddings verified (42b542915055e17c); 19 codes
